# Bayesian CUPED — Interactive Experiment Notebook

This notebook accompanies the paper **"Bayesian CUPED: Hierarchical Shrinkage for Variance Reduction in Online Experiments"** by Ayman Farahat (2026).

It allows users to:

1. **Run single-experiment analysis** (primary use case) — stratum-level and overall ATE with full inference
2. **Visualize James–Stein shrinkage** across strata
3. **Run Monte Carlo evaluation** across regimes (bias, RMSE, coverage, relative efficiency)
4. **Sensitivity analysis** — how results change with autocorrelation (ρ) and sample size (n)

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project imports
from config import (SimulationConfig, paper_default, paper_worst_case,
                    paper_best_case, paper_homogeneous, build_all_regimes)
from simulation import simulate_user_level_data
from estimators import (
    naive_dim, cuped_ols, stratified_ols, eb_cuped,
    cuped_matching_vwatt, bayesian_mcmc, run_all_fast,
    _compute_stratum_stats, EstimatorResult, StratumResult,
)
from inference import experiment_report, analyze
from evaluation import run_monte_carlo
import plots
from plots import plot_shrinkage, plot_forest, plot_summary_bars, plot_distributions

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## Configuration

The data-generating process (DGP) follows the paper:

- **Pre-period covariate:** $X_i \sim \text{Gamma}(2, 1)$
- **Stratification:** $K$ quantile-based strata from $X$
- **Outcome model:** $Y_i = \rho \, X_i + \tau_{s(i)} \, D_i + \varepsilon_i$, where $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$
- **Treatment effects:** stratum-specific $\tau_k$, with heterogeneity across strata

The autocorrelation $\rho$ controls how informative the pre-period covariate is for variance reduction.

In [ ]:
cfg = paper_default()
print(f"n = {cfg.n_users}, K = {cfg.n_strata}, \u03c1 = {cfg.rho}, \u03c3 = {cfg.sigma}")
print(f"Treatment effects: {cfg.get_stratum_effects()}")
print(f"True PATE = {cfg.true_pate():.4f}")
print(f"\u03c1(X,Y) \u2248 {cfg.effective_autocorrelation():.3f}")

rng = np.random.default_rng(cfg.random_seed)
data = simulate_user_level_data(cfg, rng)
data.head(10)

## Part 1 — Single-Experiment Inference (Primary Use Case)

Given one experiment, we produce a full report with:

- **Treatment effect on each stratum** — within-stratum DIM $\hat{\tau}_k$ and EB-shrunk $\hat{\tau}_k^{\text{EB}}$
- **VWATT** (Theorem 1): $\hat{\beta} = \sum_k \lambda_k \hat{\tau}_k$ — precision-weighted matching estimator
- **PATE** (EB CUPED, Algorithm 1): $\hat{\tau}_{\text{EB}} = \sum_k (n_k / n) \hat{\tau}_k^{\text{EB}}$ — population-weighted with shrinkage
- **Full inference** for every estimator: $\hat{\tau}$, SE, 95% CI, $t$-statistic, $p$-value

In [ ]:
stratum_df, summary_df = experiment_report(data, cfg)

print("--- Stratum-level estimates ---")
display(stratum_df)

print(f"\n--- Summary estimators ---")
display(summary_df.style.format({
    "\u03c4\u0302": "{:.6f}", "SE": "{:.6f}",
    "95% CI lower": "{:.6f}", "95% CI upper": "{:.6f}",
    "t-stat": "{:.3f}", "p-value": "{:.4f}", "n": "{:.0f}",
}).set_caption(f"Experiment Report (True PATE = {cfg.true_pate():.4f})"))

### Interpreting the report

| Row | Estimator | Estimand | Description |
|-----|-----------|----------|-------------|
| **VWATT (Theorem 1)** | CUPED matching | VWATT | Precision-weighted combination $\hat{\beta} = \sum_k \lambda_k \hat{\tau}_k$, where $\lambda_k \propto n_k p_k (1-p_k)$. Numerically equivalent to CUPED OLS when strata are the covariate bins. |
| **CUPED OLS** | OLS $Y \sim D + X$ | VWATT | Same estimand via regression adjustment on the continuous covariate. |
| **EB CUPED (E4)** | James–Stein + pop weights | PATE | Shrinks stratum estimates toward a common mean, then takes population-weighted average $\hat{\tau}_{\text{EB}} = \sum_k w_k \hat{\tau}_k^{\text{EB}}$. |
| **Naive DIM** | $\bar{Y}_T - \bar{Y}_C$ | ATE | Simple benchmark with no covariate adjustment. |

### James–Stein Shrinkage Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_shrinkage(stratum_df, true_effects=cfg.get_stratum_effects(), ax=ax)
ax.set_title("James\u2013Stein Shrinkage: Raw \u2192 EB Estimates")
plt.tight_layout()
plt.show()

### Confidence Interval Forest Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_forest(summary_df, tau_true=cfg.true_pate(), ax=ax)
plt.tight_layout()
plt.show()

### Testing equality of stratum effects

We test the null hypothesis $H_0: \tau_1 = \tau_2 = \cdots = \tau_K$ using the chi-squared statistic:

$$Q = \sum_{k=1}^{K} \frac{(\hat{\tau}_k - \hat{\mu}_\tau)^2}{s_k^2}$$

Under $H_0$, $Q \sim \chi^2_{K-1}$. A significant result indicates heterogeneous treatment effects across strata, justifying the use of stratum-specific estimation and shrinkage.

In [ ]:
res_eb, sr_eb = eb_cuped(data)
mu_hat = sr_eb.mu_tau_hat
Q = np.sum((sr_eb.tau_k_raw - mu_hat)**2 / sr_eb.s2_k)
df_chi = len(sr_eb.tau_k_raw) - 1
from scipy import stats as sp_stats
p_heterogeneity = 1 - sp_stats.chi2.cdf(Q, df_chi)
print(f"Q statistic = {Q:.3f} (df = {df_chi})")
print(f"p-value for H_0: \u03c4_1 = ... = \u03c4_K: {p_heterogeneity:.4f}")
if p_heterogeneity < 0.05:
    print("\u2192 Reject H_0: significant heterogeneity across strata")
else:
    print("\u2192 Fail to reject H_0: no significant heterogeneity detected")

### Using with your own data

You can use `experiment_report` directly with your own DataFrame. The only requirement is that it has the following columns:

| Column | Type | Description |
|--------|------|-------------|
| `D` | int (0/1) | Treatment indicator |
| `X` | float | Pre-period covariate |
| `Y` | float | Outcome |
| `stratum` | int (≥ 0) | Stratum assignment |

```python
# your_df must have columns: D (0/1), X (pre-period), Y (outcome), stratum (int >= 0)
# stratum_df, summary_df = experiment_report(your_df)
```

## Part 2 — Monte Carlo Evaluation

In [ ]:
B = 300
print(f"Running {B} Monte Carlo replications...")
mc_summary = run_monte_carlo(cfg, B=B, seed=cfg.random_seed, progress=True)
display(mc_summary.style.format("{:.4f}").set_caption(
    f"Monte Carlo Summary (B={B}, True PATE={cfg.true_pate():.4f})"))

In [ ]:
fig = plot_summary_bars(mc_summary, metrics=["RMSE", "Bias", "Coverage", "RE vs Naive"])
plt.show()

### Sampling distributions

In [ ]:
from estimators import run_all_fast

B_hist = 300
tau_hats = {label: [] for label in ["Naive DIM", "CUPED OLS", "Stratified OLS", "EB CUPED"]}
for b in range(B_hist):
    rng_b = np.random.default_rng(cfg.random_seed + b)
    d = simulate_user_level_data(cfg, rng_b)
    results = run_all_fast(d)
    for label, r in results.items():
        tau_hats[label].append(r.tau_hat)

fig, ax = plt.subplots(figsize=(10, 5))
from plots import plot_distributions
plot_distributions(tau_hats, tau_true=cfg.true_pate(), ax=ax)
plt.tight_layout()
plt.show()

## Part 3 — Full 21-Regime Comparison

The paper evaluates all estimators across **21 simulation regimes** (Table 2) spanning:

| Category | Regimes | What varies |
|----------|---------|-------------|
| Autocorrelation | baseline, high (ρ=0.9), low (ρ=0.1) | Covariate signal strength |
| Distribution | heavy tail (t₃), skewed (LogNormal) | Error shape |
| Heterogeneity | homogeneous, heterogeneous (ρ=0.7), large/small δτ | Treatment effect variation |
| Sample size | n ∈ {300, 1000, 10000} | Statistical power |
| Stratification | K ∈ {3, 10} | Granularity of matching |
| Treatment balance | p ∈ {0.20, 0.05} | Assignment imbalance |
| Noise | σ ∈ {0.3, 3.0} | Signal-to-noise ratio |
| Compound | worst case, best case | Adversarial / ideal conditions |

Run `python eval_full.py --B 300 --csv --latex` for the full sweep. Below we run a quick comparison.

In [ ]:
from eval_full import evaluate_regime, format_tables, REGIME_LABELS

all_regimes = build_all_regimes()
B_regime = 100  # increase to 300 for paper-quality results

frames = []
for name, rcfg in all_regimes.items():
    print(f"  {REGIME_LABELS.get(name, name):20s} (PATE={rcfg.true_pate():.3f}) ...", end=" ", flush=True)
    df = evaluate_regime(name, rcfg, B_regime, base_seed=42)
    frames.append(df)
    print("done")

full = pd.concat(frames, ignore_index=True)
tables = format_tables(full)

print("\n=== TABLE 3: RMSE ===")
display(tables["RMSE"].style.format("{:.4f}").set_caption("RMSE across 21 regimes"))

print("\n=== TABLE 5: Relative Efficiency vs Naive ===")
display(tables["RE"].style.format("{:.2f}").set_caption("RE vs Naive (>1 = more efficient)"))

print("\n=== Coverage (nominal = 0.95) ===")
display(tables["Coverage"].style.format("{:.3f}").set_caption("Coverage"))

In [ ]:
# Visual RMSE comparison across regimes
rmse_tbl = tables["RMSE"].drop("Mean", errors="ignore")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart: RMSE by regime for each estimator
ax = axes[0]
x = np.arange(len(rmse_tbl))
width = 0.2
for i, est in enumerate(["Naive DIM", "CUPED OLS", "EB CUPED"]):
    ax.barh(x + i * width, rmse_tbl[est], width, label=est,
            color=plots.COLORS.get(est, "#333"), alpha=0.85)
ax.set_yticks(x + width)
ax.set_yticklabels(rmse_tbl.index, fontsize=7)
ax.set_xlabel("RMSE")
ax.set_title("RMSE by Regime")
ax.legend(fontsize=8)
ax.invert_yaxis()
ax.grid(True, axis="x", alpha=0.3)

# Relative efficiency
re_tbl = tables["RE"].drop("Mean", errors="ignore")
ax2 = axes[1]
for est in ["CUPED OLS", "EB CUPED"]:
    ax2.plot(re_tbl[est].values, re_tbl.index, "o-", label=est,
             color=plots.COLORS.get(est, "#333"), markersize=5)
ax2.axvline(1.0, color="#999", linestyle="--", lw=1)
ax2.set_xlabel("Relative Efficiency vs Naive")
ax2.set_title("RE across Regimes")
ax2.legend(fontsize=8)
ax2.invert_yaxis()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Part 4 — Sensitivity Analysis

In [ ]:
rho_values = [0.1, 0.3, 0.5, 0.7, 0.9]
rmse_by_rho = {label: [] for label in ["Naive DIM", "CUPED OLS", "EB CUPED"]}

for rho in rho_values:
    c = paper_default()
    c.rho = rho
    res = run_monte_carlo(c, B=100, seed=42, progress=False)
    for label in rmse_by_rho:
        rmse_by_rho[label].append(res.loc[label, "RMSE"])

fig, ax = plt.subplots(figsize=(8, 5))
for label, vals in rmse_by_rho.items():
    ax.plot(rho_values, vals, "o-", label=label, color=plots.COLORS.get(label, "#333"))
ax.set_xlabel("Autocorrelation (\u03c1)")
ax.set_ylabel("RMSE")
ax.set_title("Sensitivity of RMSE to Autocorrelation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
n_values = [300, 500, 1000, 2000, 5000, 10000]
rmse_by_n = {label: [] for label in ["Naive DIM", "CUPED OLS", "EB CUPED"]}

for n_val in n_values:
    c = paper_default()
    c.n_users = n_val
    res = run_monte_carlo(c, B=100, seed=42, progress=False)
    for label in rmse_by_n:
        rmse_by_n[label].append(res.loc[label, "RMSE"])

fig, ax = plt.subplots(figsize=(8, 5))
for label, vals in rmse_by_n.items():
    ax.plot(n_values, vals, "o-", label=label, color=plots.COLORS.get(label, "#333"))
ax.set_xlabel("Sample size (n)")
ax.set_ylabel("RMSE")
ax.set_title("Sensitivity of RMSE to Sample Size")
ax.set_xscale("log")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 5 — Bayesian MCMC (Optional)

The Bayesian MCMC estimator (E5) fits a full hierarchical model:

$$\tau_k \sim \mathcal{N}(\mu_\tau, \sigma_\tau^2), \quad Y_{ik} \sim \mathcal{N}(\alpha_k + \tau_k D_{ik}, \sigma_y^2)$$

**Note:** MCMC is slow (~2–5 minutes). Uncomment the cell below only if you want full posterior inference.

In [ ]:
# Uncomment to run MCMC (takes ~2-5 minutes)
# res_mcmc, mcmc_detail = bayesian_mcmc(data, draws=2000, tune=1000, chains=4)
# print(f"Bayesian MCMC pop-weighted ATE: {res_mcmc.tau_hat:.4f} \u00b1 {res_mcmc.se:.4f}")
# print(f"\u03bc_\u03c4 (global): {mcmc_detail['mu_tau_post_mean']:.4f}")
# print(f"\u03c3_\u03c4 (heterogeneity): {mcmc_detail['sigma_tau_post_mean']:.4f}")
# print(f"Shrinkage B: {mcmc_detail['shrinkage_post_mean']:.4f}")
# print(f"\nStratum-level posterior means: {mcmc_detail['tau_k_post_mean']}")